In [1]:

"""
DECODELABS - DATA ANALYTICS PROJECT 3
PROJECT: SQL DATA ANALYSIS

Objective:
Analyze business data using SQL queries.
"""

import sqlite3
import csv
from pathlib import Path

# ============================================================
# 1. PROJECT SETUP
# ============================================================

OUTPUT_DIR = Path("project3_output")
OUTPUT_DIR.mkdir(exist_ok=True)

DB_PATH = OUTPUT_DIR / "decodelabs_sales.db"

# Create a fresh database each time
if DB_PATH.exists():
    DB_PATH.unlink()

connection = sqlite3.connect(DB_PATH)
cursor = connection.cursor()

print("=" * 65)
print("DECODELABS - PROJECT 3: SQL DATA ANALYSIS")
print("=" * 65)


# ============================================================
# 2. CREATE DATABASE TABLES
# ============================================================

cursor.execute("""
CREATE TABLE employees (
    employee_id INTEGER PRIMARY KEY,
    employee_name TEXT NOT NULL,
    role TEXT NOT NULL,
    department TEXT NOT NULL,
    salary REAL NOT NULL,
    city TEXT NOT NULL
)
""")

cursor.execute("""
CREATE TABLE sales (
    sale_id INTEGER PRIMARY KEY,
    employee_id INTEGER NOT NULL,
    sale_month TEXT NOT NULL,
    product TEXT NOT NULL,
    category TEXT NOT NULL,
    quantity INTEGER NOT NULL,
    unit_price REAL NOT NULL,
    region TEXT NOT NULL,
    FOREIGN KEY (employee_id)
    REFERENCES employees(employee_id)
)
""")


# ============================================================
# 3. INSERT SAMPLE DATA
# ============================================================

employees = [
    (1, "Ayesha Khan", "Developer", "Technology", 75000, "Lahore"),
    (2, "Ali Ahmed", "Developer", "Technology", 68000, "Karachi"),
    (3, "Sara Malik", "Marketer", "Marketing", 58000, "Lahore"),
    (4, "Hamza Raza", "Designer", "Design", 62000, "Islamabad"),
    (5, "Fatima Noor", "Marketer", "Marketing", 54000, "Karachi"),
    (6, "Usman Tariq", "Developer", "Technology", 82000, "Islamabad"),
    (7, "Zara Sheikh", "Designer", "Design", 60000, "Lahore"),
    (8, "Bilal Hussain", "Analyst", "Analytics", 71000, "Karachi"),
    (9, "Hira Aslam", "Analyst", "Analytics", 78000, "Lahore"),
    (10, "Danish Iqbal", "Marketer", "Marketing", 51000, "Islamabad")
]

sales = [
    (1, 1, "2026-01", "Laptop", "Electronics", 2, 120000, "North"),
    (2, 2, "2026-01", "Headphones", "Electronics", 5, 8000, "South"),
    (3, 3, "2026-01", "Office Chair", "Furniture", 3, 18000, "North"),
    (4, 4, "2026-02", "Desk Lamp", "Furniture", 4, 4500, "Central"),
    (5, 5, "2026-02", "Keyboard", "Electronics", 6, 3500, "South"),
    (6, 6, "2026-02", "Laptop", "Electronics", 1, 125000, "Central"),
    (7, 7, "2026-03", "Office Chair", "Furniture", 2, 19000, "North"),
    (8, 8, "2026-03", "Monitor", "Electronics", 3, 42000, "South"),
    (9, 9, "2026-03", "Desk Lamp", "Furniture", 8, 4000, "North"),
    (10, 10, "2026-03", "Keyboard", "Electronics", 10, 3200, "Central"),
    (11, 1, "2026-04", "Monitor", "Electronics", 2, 40000, "North"),
    (12, 2, "2026-04", "Office Chair", "Furniture", 4, 17500, "South"),
    (13, 3, "2026-04", "Laptop", "Electronics", 1, 118000, "North"),
    (14, 6, "2026-04", "Headphones", "Electronics", 7, 7500, "Central"),
    (15, 8, "2026-04", "Desk Lamp", "Furniture", 5, 4200, "South")
]

cursor.executemany(
    "INSERT INTO employees VALUES (?, ?, ?, ?, ?, ?)",
    employees
)

cursor.executemany(
    "INSERT INTO sales VALUES (?, ?, ?, ?, ?, ?, ?, ?)",
    sales
)

connection.commit()


# ============================================================
# 4. FUNCTION TO EXECUTE SQL QUERIES
# ============================================================

def run_query(title, sql, parameters=(), export_name=None):

    print("\n" + "-" * 65)
    print(title)
    print("-" * 65)

    cursor.execute(sql, parameters)

    rows = cursor.fetchall()

    columns = [
        description[0]
        for description in cursor.description
    ]

    print(" | ".join(columns))
    print("-" * 65)

    for row in rows:
        print(" | ".join(str(value) for value in row))

    print("\nTotal rows:", len(rows))

    # Export results to CSV
    if export_name:

        file_path = OUTPUT_DIR / export_name

        with open(
            file_path,
            "w",
            newline="",
            encoding="utf-8"
        ) as file:

            writer = csv.writer(file)
            writer.writerow(columns)
            writer.writerows(rows)

        print("CSV saved:", file_path)

    return rows


# ============================================================
# 5. SELECT QUERY
# ============================================================

run_query(
    "QUERY 1: Display All Employees",

    """
    SELECT *
    FROM employees
    """,

    export_name="01_all_employees.csv"
)


# ============================================================
# 6. WHERE CLAUSE
# ============================================================

run_query(
    "QUERY 2: Find Developers",

    """
    SELECT employee_name, role, salary
    FROM employees
    WHERE role = 'Developer'
    """
)


run_query(
    "QUERY 3: Employees Earning More Than 60000",

    """
    SELECT employee_name, salary
    FROM employees
    WHERE salary > ?
    ORDER BY salary DESC
    """,

    (60000,)
)


run_query(
    "QUERY 4: Find Products Containing Desk",

    """
    SELECT sale_id, product, category, quantity
    FROM sales
    WHERE product LIKE '%Desk%'
    """
)


# ============================================================
# 7. ORDER BY
# ============================================================

run_query(
    "QUERY 5: Sort Employees by Salary",

    """
    SELECT employee_name, salary
    FROM employees
    ORDER BY salary DESC
    """
)


# ============================================================
# 8. AGGREGATE FUNCTIONS
# COUNT(), SUM(), AVG()
# ============================================================

run_query(
    "QUERY 6: Total Number of Employees",

    """
    SELECT COUNT(*) AS total_employees
    FROM employees
    """
)


run_query(
    "QUERY 7: Total Sales Revenue",

    """
    SELECT SUM(quantity * unit_price) AS total_revenue
    FROM sales
    """
)


run_query(
    "QUERY 8: Average Employee Salary",

    """
    SELECT ROUND(AVG(salary), 2) AS average_salary
    FROM employees
    """
)


# ============================================================
# 9. GROUP BY
# ============================================================

run_query(
    "QUERY 9: Employees in Each Job Role",

    """
    SELECT
        role,
        COUNT(*) AS employee_count
    FROM employees
    GROUP BY role
    ORDER BY employee_count DESC
    """,

    export_name="02_employees_by_role.csv"
)


run_query(
    "QUERY 10: Revenue by Product Category",

    """
    SELECT
        category,
        SUM(quantity * unit_price) AS total_revenue
    FROM sales
    GROUP BY category
    ORDER BY total_revenue DESC
    """,

    export_name="03_revenue_by_category.csv"
)


run_query(
    "QUERY 11: Revenue by Region",

    """
    SELECT
        region,
        SUM(quantity * unit_price) AS total_revenue
    FROM sales
    GROUP BY region
    ORDER BY total_revenue DESC
    """
)


# ============================================================
# 10. HAVING CLAUSE
# ============================================================

run_query(
    "QUERY 12: Categories with Revenue Above 100000",

    """
    SELECT
        category,
        SUM(quantity * unit_price) AS total_revenue
    FROM sales
    GROUP BY category
    HAVING SUM(quantity * unit_price) > 100000
    ORDER BY total_revenue DESC
    """
)


# ============================================================
# 11. INNER JOIN
# ============================================================

run_query(
    "QUERY 13: Sales with Employee Information",

    """
    SELECT
        s.sale_id,
        e.employee_name,
        e.role,
        s.product,
        s.quantity,
        s.unit_price,
        (s.quantity * s.unit_price) AS revenue

    FROM sales AS s

    INNER JOIN employees AS e

    ON s.employee_id = e.employee_id

    ORDER BY revenue DESC
    """,

    export_name="04_sales_with_employees.csv"
)


# ============================================================
# 12. MONTHLY REVENUE ANALYSIS
# ============================================================

run_query(
    "QUERY 14: Monthly Revenue Analysis",

    """
    SELECT
        sale_month,
        SUM(quantity * unit_price) AS monthly_revenue,
        COUNT(*) AS number_of_sales

    FROM sales

    GROUP BY sale_month

    ORDER BY sale_month
    """,

    export_name="05_monthly_revenue.csv"
)


# ============================================================
# 13. PERCENTAGE CONTRIBUTION
# ============================================================

run_query(
    "QUERY 15: Category Percentage Contribution",

    """
    SELECT
        category,

        SUM(quantity * unit_price) AS category_revenue,

        ROUND(
            100.0 * SUM(quantity * unit_price) /
            (SELECT SUM(quantity * unit_price) FROM sales),
            2
        ) AS revenue_percentage

    FROM sales

    GROUP BY category

    ORDER BY category_revenue DESC
    """,

    export_name="06_category_contribution.csv"
)


# ============================================================
# 14. SQL EXECUTION ORDER DEMONSTRATION
# ============================================================

run_query(
    "QUERY 16: SQL Logical Execution Order",

    """
    SELECT
        region,
        SUM(quantity * unit_price) AS revenue

    FROM sales

    WHERE sale_month >= '2026-02'

    GROUP BY region

    HAVING SUM(quantity * unit_price) > 50000

    ORDER BY revenue DESC
    """
)


# ============================================================
# 15. BUSINESS INSIGHTS
# ============================================================

print("\n" + "=" * 65)
print("BUSINESS INSIGHTS")
print("=" * 65)

cursor.execute("SELECT COUNT(*) FROM employees")
employee_count = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM sales")
sales_count = cursor.fetchone()[0]

cursor.execute(
    "SELECT SUM(quantity * unit_price) FROM sales"
)
total_revenue = cursor.fetchone()[0]

cursor.execute("SELECT AVG(salary) FROM employees")
average_salary = cursor.fetchone()[0]

cursor.execute("""
    SELECT category,
           SUM(quantity * unit_price) AS revenue

    FROM sales

    GROUP BY category

    ORDER BY revenue DESC

    LIMIT 1
""")

top_category, top_category_revenue = cursor.fetchone()

cursor.execute("""
    SELECT region,
           SUM(quantity * unit_price) AS revenue

    FROM sales

    GROUP BY region

    ORDER BY revenue DESC

    LIMIT 1
""")

top_region, top_region_revenue = cursor.fetchone()

print("1. Total Employees:", employee_count)
print("2. Total Sales Transactions:", sales_count)

print(
    "3. Total Revenue:",
    f"{total_revenue:,.2f}"
)

print(
    "4. Average Salary:",
    f"{average_salary:,.2f}"
)

print(
    "5. Top Product Category:",
    top_category,
    f"({top_category_revenue:,.2f})"
)

print(
    "6. Top Sales Region:",
    top_region,
    f"({top_region_revenue:,.2f})"
)


# ============================================================
# 16. PROJECT COMPLETION
# ============================================================

connection.close()

print("\n" + "=" * 65)
print("PROJECT COMPLETED SUCCESSFULLY!")
print("=" * 65)

print("Database and CSV files saved in:")
print(OUTPUT_DIR.resolve())


DECODELABS - PROJECT 3: SQL DATA ANALYSIS

-----------------------------------------------------------------
QUERY 1: Display All Employees
-----------------------------------------------------------------
employee_id | employee_name | role | department | salary | city
-----------------------------------------------------------------
1 | Ayesha Khan | Developer | Technology | 75000.0 | Lahore
2 | Ali Ahmed | Developer | Technology | 68000.0 | Karachi
3 | Sara Malik | Marketer | Marketing | 58000.0 | Lahore
4 | Hamza Raza | Designer | Design | 62000.0 | Islamabad
5 | Fatima Noor | Marketer | Marketing | 54000.0 | Karachi
6 | Usman Tariq | Developer | Technology | 82000.0 | Islamabad
7 | Zara Sheikh | Designer | Design | 60000.0 | Lahore
8 | Bilal Hussain | Analyst | Analytics | 71000.0 | Karachi
9 | Hira Aslam | Analyst | Analytics | 78000.0 | Lahore
10 | Danish Iqbal | Marketer | Marketing | 51000.0 | Islamabad

Total rows: 10
CSV saved: project3_output/01_all_employees.csv

----------